In [ ]:
%sql
-- Databricks SQL script: Comprehensive test suite for forecast ERP logic (f_inv_movmnt report)
-- Purpose: Validate forecast ERP SQL logic, mapping, data quality, and error handling for f_inv_movmnt report generation
-- Author: Giang Nguyen
-- Date: 2025-09-29
-- Description: This script tests the forecast ERP SQL logic for f_inv_movmnt report generation per inv_txn_mapping.xlsx. It covers schema validation, data type enforcement, NULL and error handling, valid/invalid value checks, duplicate handling, and data quality rules. All assertions use Databricks SQL patterns and CTEs. No temp views or temp tables are used.

USE CATALOG purgo_databricks;
USE purgo_playground;

/*-----------------------------------------------------------------------------
  SECTION: Test Data Setup
  - CTE: test_forecast_erp_data
  - Provides comprehensive test data for all test scenarios
-----------------------------------------------------------------------------*/
WITH test_forecast_erp_data AS (
    SELECT 'TXN001' AS txn_id, CAST(8.0 AS DECIMAL(38,2)) AS allocated_qty, CAST(20240601 AS DECIMAL(38,0)) AS delivery_dt, CAST(20240605 AS DECIMAL(38,0)) AS sched_dt, TRUE AS flag_key
    UNION ALL SELECT 'TXN002', CAST(0.0 AS DECIMAL(38,2)), CAST(20240602 AS DECIMAL(38,0)), CAST(20240606 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN003', CAST(7.0 AS DECIMAL(38,2)), CAST(20240603 AS DECIMAL(38,0)), CAST(20240607 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN004', CAST(6.0 AS DECIMAL(38,2)), CAST(20240604 AS DECIMAL(38,0)), CAST(20240608 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN005', CAST(2.0 AS DECIMAL(38,2)), CAST(20240605 AS DECIMAL(38,0)), CAST(20240609 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN006', NULL, CAST(20240610 AS DECIMAL(38,0)), CAST(20240611 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN007', CAST(5.0 AS DECIMAL(38,2)), NULL, CAST(20240612 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN008', CAST(4.0 AS DECIMAL(38,2)), CAST(20240613 AS DECIMAL(38,0)), NULL, FALSE
    UNION ALL SELECT 'TXN009', CAST(3.0 AS DECIMAL(38,2)), CAST(20240614 AS DECIMAL(38,0)), CAST(20240615 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN010', CAST(NULL AS DECIMAL(38,2)), CAST(20240616 AS DECIMAL(38,0)), CAST(20240617 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN011', CAST(5.0 AS DECIMAL(38,2)), CAST(202406 AS DECIMAL(38,0)), CAST(20240618 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN012', CAST(4.0 AS DECIMAL(38,2)), CAST(20240619 AS DECIMAL(38,0)), CAST(202406 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN013', CAST(3.0 AS DECIMAL(38,2)), CAST(20240620 AS DECIMAL(38,0)), CAST(20240621 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN014', CAST(-1.0 AS DECIMAL(38,2)), CAST(20240622 AS DECIMAL(38,0)), CAST(20240623 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN015', CAST(99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240624 AS DECIMAL(38,0)), CAST(20240625 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN016', CAST(1.0 AS DECIMAL(38,2)), CAST(19000101 AS DECIMAL(38,0)), CAST(20240626 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN017', CAST(2.0 AS DECIMAL(38,2)), CAST(99991231 AS DECIMAL(38,0)), CAST(20240627 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN018', CAST(3.0 AS DECIMAL(38,2)), CAST(20240628 AS DECIMAL(38,0)), CAST(19000101 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN019', CAST(4.0 AS DECIMAL(38,2)), CAST(20240629 AS DECIMAL(38,0)), CAST(99991231 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT NULL, NULL, NULL, NULL, NULL
    UNION ALL SELECT 'TXN_!@#', CAST(5.0 AS DECIMAL(38,2)), CAST(20240630 AS DECIMAL(38,0)), CAST(20240701 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN_日本語', CAST(6.0 AS DECIMAL(38,2)), CAST(20240702 AS DECIMAL(38,0)), CAST(20240703 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN_😀', CAST(7.0 AS DECIMAL(38,2)), CAST(20240704 AS DECIMAL(38,0)), CAST(20240705 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN021', NULL, CAST(NULL AS DECIMAL(38,0)), CAST(20240706 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN022', CAST(8.0 AS DECIMAL(38,2)), CAST(20240707 AS DECIMAL(38,0)), CAST(NULL AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN023', CAST(9.0 AS DECIMAL(38,2)), CAST(20240708 AS DECIMAL(38,0)), CAST(20240709 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN001', CAST(10.0 AS DECIMAL(38,2)), CAST(20240710 AS DECIMAL(38,0)), CAST(20240711 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN002', CAST(0.0 AS DECIMAL(38,2)), CAST(20240712 AS DECIMAL(38,0)), CAST(20240713 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN025', CAST(NULL AS DECIMAL(38,2)), CAST(20240714 AS DECIMAL(38,0)), CAST(20240715 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN026', CAST(11.0 AS DECIMAL(38,2)), CAST(-20240716 AS DECIMAL(38,0)), CAST(20240717 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN027', CAST(12.0 AS DECIMAL(38,2)), CAST(20240718 AS DECIMAL(38,0)), CAST(-20240719 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN028', CAST(0.0 AS DECIMAL(38,2)), CAST(20240720 AS DECIMAL(38,0)), CAST(20240721 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN029', CAST(99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240722 AS DECIMAL(38,0)), CAST(20240723 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN030', CAST(-99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240724 AS DECIMAL(38,0)), CAST(20240725 AS DECIMAL(38,0)), FALSE
)

/*-----------------------------------------------------------------------------
  SECTION: Schema Validation
  - Ensures all columns exist and have correct data types
-----------------------------------------------------------------------------*/
-- Validate schema: column names and data types
SELECT
    CASE WHEN typeof(txn_id) = "STRING" THEN "PASS" ELSE "FAIL: txn_id not STRING" END AS txn_id_type_check,
    CASE WHEN typeof(allocated_qty) = "DECIMAL(38,2)" THEN "PASS" ELSE "FAIL: allocated_qty not DECIMAL(38,2)" END AS allocated_qty_type_check,
    CASE WHEN typeof(delivery_dt) = "DECIMAL(38,0)" THEN "PASS" ELSE "FAIL: delivery_dt not DECIMAL(38,0)" END AS delivery_dt_type_check,
    CASE WHEN typeof(sched_dt) = "DECIMAL(38,0)" THEN "PASS" ELSE "FAIL: sched_dt not DECIMAL(38,0)" END AS sched_dt_type_check,
    CASE WHEN typeof(flag_key) = "BOOLEAN" THEN "PASS" ELSE "FAIL: flag_key not BOOLEAN" END AS flag_key_type_check
FROM test_forecast_erp_data
LIMIT 1
;

-- Assert: All columns must pass type check
SELECT
    CASE
        WHEN typeof(txn_id) = "STRING"
         AND typeof(allocated_qty) = "DECIMAL(38,2)"
         AND typeof(delivery_dt) = "DECIMAL(38,0)"
         AND typeof(sched_dt) = "DECIMAL(38,0)"
         AND typeof(flag_key) = "BOOLEAN"
        THEN "PASS"
        ELSE "FAIL"
    END AS schema_validation_result
FROM test_forecast_erp_data
LIMIT 1
;

/*-----------------------------------------------------------------------------
  SECTION: Data Type and NULL Handling Tests
  - Validate decimal, boolean, and yyyymmdd format
  - NULLs default to 0.00 for allocated_qty, error for flag_key
-----------------------------------------------------------------------------*/
-- Validate allocated_qty: must be decimal, default 0.00 if NULL
SELECT
    COUNT(*) AS null_allocated_qty_count
FROM (
    SELECT
        txn_id,
        CASE
            WHEN allocated_qty IS NULL THEN 0.00
            ELSE allocated_qty
        END AS allocated_qty_checked
    FROM test_forecast_erp_data
) checked
WHERE allocated_qty_checked = 0.00
;

-- Validate delivery_dt and sched_dt: must be yyyymmdd (8 digits), decimal(38,0)
SELECT
    COUNT(*) AS invalid_delivery_dt_count
FROM (
    SELECT
        txn_id,
        delivery_dt,
        LENGTH(CAST(delivery_dt AS STRING)) AS delivery_dt_len
    FROM test_forecast_erp_data
) checked
WHERE delivery_dt IS NOT NULL AND delivery_dt_len <> 8
;

SELECT
    COUNT(*) AS invalid_sched_dt_count
FROM (
    SELECT
        txn_id,
        sched_dt,
        LENGTH(CAST(sched_dt AS STRING)) AS sched_dt_len
    FROM test_forecast_erp_data
) checked
WHERE sched_dt IS NOT NULL AND sched_dt_len <> 8
;

-- Validate flag_key: must be boolean (true/false), not NULL
SELECT
    COUNT(*) AS null_flag_key_count
FROM test_forecast_erp_data
WHERE flag_key IS NULL
;

/*-----------------------------------------------------------------------------
  SECTION: Data Quality Validation
  - Enforce valid values and error messages for invalid data
-----------------------------------------------------------------------------*/
-- Data quality: allocated_qty must be decimal, else error
SELECT
    txn_id,
    CASE
        WHEN allocated_qty IS NULL THEN "allocated_qty is NULL"
        WHEN typeof(allocated_qty) <> "DECIMAL(38,2)" THEN "allocated_qty must be decimal"
        ELSE NULL
    END AS error_msg
FROM test_forecast_erp_data
WHERE allocated_qty IS NULL OR typeof(allocated_qty) <> "DECIMAL(38,2)"
;

-- Data quality: delivery_dt and sched_dt must be yyyymmdd (8 digits)
SELECT
    txn_id,
    CASE
        WHEN delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8 THEN "delivery_dt must be yyyymmdd format"
        ELSE NULL
    END AS delivery_dt_error,
    CASE
        WHEN sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8 THEN "sched_dt must be yyyymmdd format"
        ELSE NULL
    END AS sched_dt_error
FROM test_forecast_erp_data
WHERE (delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8)
   OR (sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8)
;

-- Data quality: flag_key must be boolean
SELECT
    txn_id,
    CASE
        WHEN flag_key IS NULL THEN "flag_key must be boolean"
        ELSE NULL
    END AS flag_key_error
FROM test_forecast_erp_data
WHERE flag_key IS NULL
;

/*-----------------------------------------------------------------------------
  SECTION: Duplicate Handling
  - Ensure only one row per unique txn_id, sum allocated_qty for duplicates
-----------------------------------------------------------------------------*/
-- Find duplicate txn_id
SELECT
    txn_id,
    COUNT(*) AS cnt
FROM test_forecast_erp_data
WHERE txn_id IS NOT NULL
GROUP BY txn_id
HAVING cnt > 1
;

-- Validate sum of allocated_qty for duplicate txn_id
SELECT
    txn_id,
    SUM(COALESCE(allocated_qty, 0.00)) AS total_allocated_qty
FROM test_forecast_erp_data
WHERE txn_id = "TXN001"
GROUP BY txn_id
;

/*-----------------------------------------------------------------------------
  SECTION: Error Path and Data Quality Enforcement
  - Return error messages for invalid rows
-----------------------------------------------------------------------------*/
-- Error path: allocated_qty not decimal, delivery_dt/sched_dt not yyyymmdd, flag_key not boolean
SELECT
    txn_id,
    CASE
        WHEN allocated_qty IS NULL THEN "allocated_qty must be decimal"
        WHEN typeof(allocated_qty) <> "DECIMAL(38,2)" THEN "allocated_qty must be decimal"
        WHEN delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8 THEN "delivery_dt must be yyyymmdd format"
        WHEN sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8 THEN "sched_dt must be yyyymmdd format"
        WHEN flag_key IS NULL THEN "flag_key must be boolean"
        ELSE NULL
    END AS error_message
FROM test_forecast_erp_data
WHERE
    allocated_qty IS NULL
    OR typeof(allocated_qty) <> "DECIMAL(38,2)"
    OR (delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8)
    OR (sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8)
    OR flag_key IS NULL
;

/*-----------------------------------------------------------------------------
  SECTION: Output Structure Validation
  - Ensure output columns and types match mapping
-----------------------------------------------------------------------------*/
-- Validate output structure: columns and types
SELECT
    CASE
        WHEN typeof(txn_id) = "STRING"
         AND typeof(allocated_qty) = "DECIMAL(38,2)"
         AND typeof(delivery_dt) = "DECIMAL(38,0)"
         AND typeof(sched_dt) = "DECIMAL(38,0)"
         AND typeof(flag_key) = "BOOLEAN"
        THEN "PASS"
        ELSE "FAIL"
    END AS output_structure_validation
FROM test_forecast_erp_data
LIMIT 1
;

/*-----------------------------------------------------------------------------
  SECTION: Performance Test (Row Count)
  - Ensure all test rows are processed
-----------------------------------------------------------------------------*/
-- Count total test rows
SELECT COUNT(*) AS total_test_rows FROM test_forecast_erp_data
;

/*-----------------------------------------------------------------------------
  SECTION: Window Function Test
  - Use window function to rank by allocated_qty per flag_key
-----------------------------------------------------------------------------*/
-- Window function: rank by allocated_qty within flag_key
SELECT
    txn_id,
    allocated_qty,
    flag_key,
    RANK() OVER (PARTITION BY flag_key ORDER BY allocated_qty DESC NULLS LAST) AS rank_allocated
FROM test_forecast_erp_data
ORDER BY flag_key DESC, rank_allocated ASC
;

/*-----------------------------------------------------------------------------
  SECTION: Delta Lake Operation Test (MERGE/UPDATE/DELETE)
  - Test MERGE, UPDATE, DELETE on a test Delta table
-----------------------------------------------------------------------------*/
-- Create test Delta table for DML operations
CREATE TABLE IF NOT EXISTS purgo_playground.test_forecast_erp_delta (
    txn_id STRING,
    allocated_qty DECIMAL(38,2),
    delivery_dt DECIMAL(38,0),
    sched_dt DECIMAL(38,0),
    flag_key BOOLEAN,
    CONSTRAINT delivery_dt_format CHECK (LENGTH(CAST(delivery_dt AS STRING)) = 8),
    CONSTRAINT sched_dt_format CHECK (LENGTH(CAST(sched_dt AS STRING)) = 8),
    CONSTRAINT flag_key_bool CHECK (flag_key IN (TRUE, FALSE))
);

-- Insert test data (only valid rows)
INSERT INTO purgo_playground.test_forecast_erp_delta (txn_id, allocated_qty, delivery_dt, sched_dt, flag_key)
SELECT
    txn_id, COALESCE(allocated_qty, 0.00), delivery_dt, sched_dt, flag_key
FROM test_forecast_erp_data
WHERE
    allocated_qty IS NOT NULL
    AND delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) = 8
    AND sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) = 8
    AND flag_key IS NOT NULL
;

-- MERGE: Upsert a row

WITH test_forecast_erp_data AS (
    SELECT 'TXN001' AS txn_id, CAST(8.0 AS DECIMAL(38,2)) AS allocated_qty, CAST(20240601 AS DECIMAL(38,0)) AS delivery_dt, CAST(20240605 AS DECIMAL(38,0)) AS sched_dt, TRUE AS flag_key
    UNION ALL SELECT 'TXN002', CAST(0.0 AS DECIMAL(38,2)), CAST(20240602 AS DECIMAL(38,0)), CAST(20240606 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN003', CAST(7.0 AS DECIMAL(38,2)), CAST(20240603 AS DECIMAL(38,0)), CAST(20240607 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN004', CAST(6.0 AS DECIMAL(38,2)), CAST(20240604 AS DECIMAL(38,0)), CAST(20240608 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN005', CAST(2.0 AS DECIMAL(38,2)), CAST(20240605 AS DECIMAL(38,0)), CAST(20240609 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN006', NULL, CAST(20240610 AS DECIMAL(38,0)), CAST(20240611 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN007', CAST(5.0 AS DECIMAL(38,2)), NULL, CAST(20240612 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN008', CAST(4.0 AS DECIMAL(38,2)), CAST(20240613 AS DECIMAL(38,0)), NULL, FALSE
    UNION ALL SELECT 'TXN009', CAST(3.0 AS DECIMAL(38,2)), CAST(20240614 AS DECIMAL(38,0)), CAST(20240615 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN010', CAST(NULL AS DECIMAL(38,2)), CAST(20240616 AS DECIMAL(38,0)), CAST(20240617 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN011', CAST(5.0 AS DECIMAL(38,2)), CAST(202406 AS DECIMAL(38,0)), CAST(20240618 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN012', CAST(4.0 AS DECIMAL(38,2)), CAST(20240619 AS DECIMAL(38,0)), CAST(202406 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN013', CAST(3.0 AS DECIMAL(38,2)), CAST(20240620 AS DECIMAL(38,0)), CAST(20240621 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN014', CAST(-1.0 AS DECIMAL(38,2)), CAST(20240622 AS DECIMAL(38,0)), CAST(20240623 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN015', CAST(99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240624 AS DECIMAL(38,0)), CAST(20240625 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN016', CAST(1.0 AS DECIMAL(38,2)), CAST(19000101 AS DECIMAL(38,0)), CAST(20240626 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN017', CAST(2.0 AS DECIMAL(38,2)), CAST(99991231 AS DECIMAL(38,0)), CAST(20240627 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN018', CAST(3.0 AS DECIMAL(38,2)), CAST(20240628 AS DECIMAL(38,0)), CAST(19000101 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN019', CAST(4.0 AS DECIMAL(38,2)), CAST(20240629 AS DECIMAL(38,0)), CAST(99991231 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT NULL, NULL, NULL, NULL, NULL
    UNION ALL SELECT 'TXN_!@#', CAST(5.0 AS DECIMAL(38,2)), CAST(20240630 AS DECIMAL(38,0)), CAST(20240701 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN_日本語', CAST(6.0 AS DECIMAL(38,2)), CAST(20240702 AS DECIMAL(38,0)), CAST(20240703 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN_😀', CAST(7.0 AS DECIMAL(38,2)), CAST(20240704 AS DECIMAL(38,0)), CAST(20240705 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN021', NULL, CAST(NULL AS DECIMAL(38,0)), CAST(20240706 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN022', CAST(8.0 AS DECIMAL(38,2)), CAST(20240707 AS DECIMAL(38,0)), CAST(NULL AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN023', CAST(9.0 AS DECIMAL(38,2)), CAST(20240708 AS DECIMAL(38,0)), CAST(20240709 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN001', CAST(10.0 AS DECIMAL(38,2)), CAST(20240710 AS DECIMAL(38,0)), CAST(20240711 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN002', CAST(0.0 AS DECIMAL(38,2)), CAST(20240712 AS DECIMAL(38,0)), CAST(20240713 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN025', CAST(NULL AS DECIMAL(38,2)), CAST(20240714 AS DECIMAL(38,0)), CAST(20240715 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN026', CAST(11.0 AS DECIMAL(38,2)), CAST(-20240716 AS DECIMAL(38,0)), CAST(20240717 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN027', CAST(12.0 AS DECIMAL(38,2)), CAST(20240718 AS DECIMAL(38,0)), CAST(-20240719 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN028', CAST(0.0 AS DECIMAL(38,2)), CAST(20240720 AS DECIMAL(38,0)), CAST(20240721 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN029', CAST(99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240722 AS DECIMAL(38,0)), CAST(20240723 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN030', CAST(-99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240724 AS DECIMAL(38,0)), CAST(20240725 AS DECIMAL(38,0)), FALSE
)
SELECT
    CASE WHEN typeof(txn_id) = "STRING" THEN "PASS" ELSE "FAIL: txn_id not STRING" END AS txn_id_type_check,
    CASE WHEN typeof(allocated_qty) = "DECIMAL(38,2)" THEN "PASS" ELSE "FAIL: allocated_qty not DECIMAL(38,2)" END AS allocated_qty_type_check,
    CASE WHEN typeof(delivery_dt) = "DECIMAL(38,0)" THEN "PASS" ELSE "FAIL: delivery_dt not DECIMAL(38,0)" END AS delivery_dt_type_check,
    CASE WHEN typeof(sched_dt) = "DECIMAL(38,0)" THEN "PASS" ELSE "FAIL: sched_dt not DECIMAL(38,0)" END AS sched_dt_type_check,
    CASE WHEN typeof(flag_key) = "BOOLEAN" THEN "PASS" ELSE "FAIL: flag_key not BOOLEAN" END AS flag_key_type_check
FROM test_forecast_erp_data
LIMIT 1
;
SELECT
    CASE
        WHEN typeof(txn_id) = "STRING"
         AND typeof(allocated_qty) = "DECIMAL(38,2)"
         AND typeof(delivery_dt) = "DECIMAL(38,0)"
         AND typeof(sched_dt) = "DECIMAL(38,0)"
         AND typeof(flag_key) = "BOOLEAN"
        THEN "PASS"
        ELSE "FAIL"
    END AS schema_validation_result
FROM test_forecast_erp_data
LIMIT 1
;
SELECT
    COUNT(*) AS null_allocated_qty_count
FROM (
    SELECT
        txn_id,
        CASE
            WHEN allocated_qty IS NULL THEN 0.00
            ELSE allocated_qty
        END AS allocated_qty_checked
    FROM test_forecast_erp_data
) checked
WHERE allocated_qty_checked = 0.00
;
SELECT
    COUNT(*) AS invalid_delivery_dt_count
FROM (
    SELECT
        txn_id,
        delivery_dt,
        LENGTH(CAST(delivery_dt AS STRING)) AS delivery_dt_len
    FROM test_forecast_erp_data
) checked
WHERE delivery_dt IS NOT NULL AND delivery_dt_len <> 8
;
SELECT
    COUNT(*) AS invalid_sched_dt_count
FROM (
    SELECT
        txn_id,
        sched_dt,
        LENGTH(CAST(sched_dt AS STRING)) AS sched_dt_len
    FROM test_forecast_erp_data
) checked
WHERE sched_dt IS NOT NULL AND sched_dt_len <> 8
;
SELECT
    COUNT(*) AS null_flag_key_count
FROM test_forecast_erp_data
WHERE flag_key IS NULL
;
SELECT
    txn_id,
    CASE
        WHEN allocated_qty IS NULL THEN "allocated_qty is NULL"
        WHEN typeof(allocated_qty) <> "DECIMAL(38,2)" THEN "allocated_qty must be decimal"
        ELSE NULL
    END AS error_msg
FROM test_forecast_erp_data
WHERE allocated_qty IS NULL OR typeof(allocated_qty) <> "DECIMAL(38,2)"
;
SELECT
    txn_id,
    CASE
        WHEN delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8 THEN "delivery_dt must be yyyymmdd format"
        ELSE NULL
    END AS delivery_dt_error,
    CASE
        WHEN sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8 THEN "sched_dt must be yyyymmdd format"
        ELSE NULL
    END AS sched_dt_error
FROM test_forecast_erp_data
WHERE (delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8)
   OR (sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8)
;
SELECT
    txn_id,
    CASE
        WHEN flag_key IS NULL THEN "flag_key must be boolean"
        ELSE NULL
    END AS flag_key_error
FROM test_forecast_erp_data
WHERE flag_key IS NULL
;
SELECT
    txn_id,
    COUNT(*) AS cnt
FROM test_forecast_erp_data
WHERE txn_id IS NOT NULL
GROUP BY txn_id
HAVING cnt > 1
;
SELECT
    txn_id,
    SUM(COALESCE(allocated_qty, 0.00)) AS total_allocated_qty
FROM test_forecast_erp_data
WHERE txn_id = "TXN001"
GROUP BY txn_id
;
SELECT
    txn_id,
    CASE
        WHEN allocated_qty IS NULL THEN "allocated_qty must be decimal"
        WHEN typeof(allocated_qty) <> "DECIMAL(38,2)" THEN "allocated_qty must be decimal"
        WHEN delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8 THEN "delivery_dt must be yyyymmdd format"
        WHEN sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8 THEN "sched_dt must be yyyymmdd format"
        WHEN flag_key IS NULL THEN "flag_key must be boolean"
        ELSE NULL
    END AS error_message
FROM test_forecast_erp_data
WHERE
    allocated_qty IS NULL
    OR typeof(allocated_qty) <> "DECIMAL(38,2)"
    OR (delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8)
    OR (sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8)
    OR flag_key IS NULL
;
SELECT
    CASE
        WHEN typeof(txn_id) = "STRING"
         AND typeof(allocated_qty) = "DECIMAL(38,2)"
         AND typeof(delivery_dt) = "DECIMAL(38,0)"
         AND typeof(sched_dt) = "DECIMAL(38,0)"
         AND typeof(flag_key) = "BOOLEAN"
        THEN "PASS"
        ELSE "FAIL"
    END AS output_structure_validation
FROM test_forecast_erp_data
LIMIT 1
;
SELECT COUNT(*) AS total_test_rows FROM test_forecast_erp_data
;
SELECT
    txn_id,
    allocated_qty,
    flag_key,
    RANK() OVER (PARTITION BY flag_key ORDER BY allocated_qty DESC NULLS LAST) AS rank_allocated
FROM test_forecast_erp_data
ORDER BY flag_key DESC, rank_allocated ASC
;
CREATE TABLE IF NOT EXISTS purgo_playground.test_forecast_erp_delta (
    txn_id STRING,
    allocated_qty DECIMAL(38,2),
    delivery_dt DECIMAL(38,0),
    sched_dt DECIMAL(38,0),
    flag_key BOOLEAN,
    CONSTRAINT delivery_dt_format CHECK (LENGTH(CAST(delivery_dt AS STRING)) = 8),
    CONSTRAINT sched_dt_format CHECK (LENGTH(CAST(sched_dt AS STRING)) = 8),
    CONSTRAINT flag_key_bool CHECK (flag_key IN (TRUE, FALSE))
);

MERGE INTO purgo_playground.test_forecast_erp_delta AS target
USING (SELECT "TXN999" AS txn_id, 99.99 AS allocated_qty, 20250101 AS delivery_dt, 20250102 AS sched_dt, TRUE AS flag_key) AS source
ON target.txn_id = source.txn_id
WHEN MATCHED THEN
  UPDATE SET allocated_qty = source.allocated_qty
WHEN NOT MATCHED THEN
  INSERT (txn_id, allocated_qty, delivery_dt, sched_dt, flag_key)
  VALUES (source.txn_id, source.allocated_qty, source.delivery_dt, source.sched_dt, source.flag_key)
;

-- UPDATE: Set allocated_qty to 0.00 for a specific txn_id
UPDATE purgo_playground.test_forecast_erp_delta
SET allocated_qty = 0.00
WHERE txn_id = "TXN001"
;

-- DELETE: Remove a specific test row

WITH test_forecast_erp_data AS (
    SELECT 'TXN001' AS txn_id, CAST(8.0 AS DECIMAL(38,2)) AS allocated_qty, CAST(20240601 AS DECIMAL(38,0)) AS delivery_dt, CAST(20240605 AS DECIMAL(38,0)) AS sched_dt, TRUE AS flag_key
    UNION ALL SELECT 'TXN002', CAST(0.0 AS DECIMAL(38,2)), CAST(20240602 AS DECIMAL(38,0)), CAST(20240606 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN003', CAST(7.0 AS DECIMAL(38,2)), CAST(20240603 AS DECIMAL(38,0)), CAST(20240607 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN004', CAST(6.0 AS DECIMAL(38,2)), CAST(20240604 AS DECIMAL(38,0)), CAST(20240608 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN005', CAST(2.0 AS DECIMAL(38,2)), CAST(20240605 AS DECIMAL(38,0)), CAST(20240609 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN006', NULL, CAST(20240610 AS DECIMAL(38,0)), CAST(20240611 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN007', CAST(5.0 AS DECIMAL(38,2)), NULL, CAST(20240612 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN008', CAST(4.0 AS DECIMAL(38,2)), CAST(20240613 AS DECIMAL(38,0)), NULL, FALSE
    UNION ALL SELECT 'TXN009', CAST(3.0 AS DECIMAL(38,2)), CAST(20240614 AS DECIMAL(38,0)), CAST(20240615 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN010', CAST(NULL AS DECIMAL(38,2)), CAST(20240616 AS DECIMAL(38,0)), CAST(20240617 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN011', CAST(5.0 AS DECIMAL(38,2)), CAST(202406 AS DECIMAL(38,0)), CAST(20240618 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN012', CAST(4.0 AS DECIMAL(38,2)), CAST(20240619 AS DECIMAL(38,0)), CAST(202406 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN013', CAST(3.0 AS DECIMAL(38,2)), CAST(20240620 AS DECIMAL(38,0)), CAST(20240621 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN014', CAST(-1.0 AS DECIMAL(38,2)), CAST(20240622 AS DECIMAL(38,0)), CAST(20240623 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN015', CAST(99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240624 AS DECIMAL(38,0)), CAST(20240625 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN016', CAST(1.0 AS DECIMAL(38,2)), CAST(19000101 AS DECIMAL(38,0)), CAST(20240626 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN017', CAST(2.0 AS DECIMAL(38,2)), CAST(99991231 AS DECIMAL(38,0)), CAST(20240627 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN018', CAST(3.0 AS DECIMAL(38,2)), CAST(20240628 AS DECIMAL(38,0)), CAST(19000101 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN019', CAST(4.0 AS DECIMAL(38,2)), CAST(20240629 AS DECIMAL(38,0)), CAST(99991231 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT NULL, NULL, NULL, NULL, NULL
    UNION ALL SELECT 'TXN_!@#', CAST(5.0 AS DECIMAL(38,2)), CAST(20240630 AS DECIMAL(38,0)), CAST(20240701 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN_日本語', CAST(6.0 AS DECIMAL(38,2)), CAST(20240702 AS DECIMAL(38,0)), CAST(20240703 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN_😀', CAST(7.0 AS DECIMAL(38,2)), CAST(20240704 AS DECIMAL(38,0)), CAST(20240705 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN021', NULL, CAST(NULL AS DECIMAL(38,0)), CAST(20240706 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN022', CAST(8.0 AS DECIMAL(38,2)), CAST(20240707 AS DECIMAL(38,0)), CAST(NULL AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN023', CAST(9.0 AS DECIMAL(38,2)), CAST(20240708 AS DECIMAL(38,0)), CAST(20240709 AS DECIMAL(38,0)), NULL
    UNION ALL SELECT 'TXN001', CAST(10.0 AS DECIMAL(38,2)), CAST(20240710 AS DECIMAL(38,0)), CAST(20240711 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN002', CAST(0.0 AS DECIMAL(38,2)), CAST(20240712 AS DECIMAL(38,0)), CAST(20240713 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN025', CAST(NULL AS DECIMAL(38,2)), CAST(20240714 AS DECIMAL(38,0)), CAST(20240715 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN026', CAST(11.0 AS DECIMAL(38,2)), CAST(-20240716 AS DECIMAL(38,0)), CAST(20240717 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN027', CAST(12.0 AS DECIMAL(38,2)), CAST(20240718 AS DECIMAL(38,0)), CAST(-20240719 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN028', CAST(0.0 AS DECIMAL(38,2)), CAST(20240720 AS DECIMAL(38,0)), CAST(20240721 AS DECIMAL(38,0)), FALSE
    UNION ALL SELECT 'TXN029', CAST(99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240722 AS DECIMAL(38,0)), CAST(20240723 AS DECIMAL(38,0)), TRUE
    UNION ALL SELECT 'TXN030', CAST(-99999999999999999999.99 AS DECIMAL(38,2)), CAST(20240724 AS DECIMAL(38,0)), CAST(20240725 AS DECIMAL(38,0)), FALSE
)
SELECT
    CASE WHEN typeof(txn_id) = "STRING" THEN "PASS" ELSE "FAIL: txn_id not STRING" END AS txn_id_type_check,
    CASE WHEN typeof(allocated_qty) = "DECIMAL(38,2)" THEN "PASS" ELSE "FAIL: allocated_qty not DECIMAL(38,2)" END AS allocated_qty_type_check,
    CASE WHEN typeof(delivery_dt) = "DECIMAL(38,0)" THEN "PASS" ELSE "FAIL: delivery_dt not DECIMAL(38,0)" END AS delivery_dt_type_check,
    CASE WHEN typeof(sched_dt) = "DECIMAL(38,0)" THEN "PASS" ELSE "FAIL: sched_dt not DECIMAL(38,0)" END AS sched_dt_type_check,
    CASE WHEN typeof(flag_key) = "BOOLEAN" THEN "PASS" ELSE "FAIL: flag_key not BOOLEAN" END AS flag_key_type_check
FROM test_forecast_erp_data
LIMIT 1
;
SELECT
    CASE
        WHEN typeof(txn_id) = "STRING"
         AND typeof(allocated_qty) = "DECIMAL(38,2)"
         AND typeof(delivery_dt) = "DECIMAL(38,0)"
         AND typeof(sched_dt) = "DECIMAL(38,0)"
         AND typeof(flag_key) = "BOOLEAN"
        THEN "PASS"
        ELSE "FAIL"
    END AS schema_validation_result
FROM test_forecast_erp_data
LIMIT 1
;
SELECT
    COUNT(*) AS null_allocated_qty_count
FROM (
    SELECT
        txn_id,
        CASE
            WHEN allocated_qty IS NULL THEN 0.00
            ELSE allocated_qty
        END AS allocated_qty_checked
    FROM test_forecast_erp_data
) checked
WHERE allocated_qty_checked = 0.00
;
SELECT
    COUNT(*) AS invalid_delivery_dt_count
FROM (
    SELECT
        txn_id,
        delivery_dt,
        LENGTH(CAST(delivery_dt AS STRING)) AS delivery_dt_len
    FROM test_forecast_erp_data
) checked
WHERE delivery_dt IS NOT NULL AND delivery_dt_len <> 8
;
SELECT
    COUNT(*) AS invalid_sched_dt_count
FROM (
    SELECT
        txn_id,
        sched_dt,
        LENGTH(CAST(sched_dt AS STRING)) AS sched_dt_len
    FROM test_forecast_erp_data
) checked
WHERE sched_dt IS NOT NULL AND sched_dt_len <> 8
;
SELECT
    COUNT(*) AS null_flag_key_count
FROM test_forecast_erp_data
WHERE flag_key IS NULL
;
SELECT
    txn_id,
    CASE
        WHEN allocated_qty IS NULL THEN "allocated_qty is NULL"
        WHEN typeof(allocated_qty) <> "DECIMAL(38,2)" THEN "allocated_qty must be decimal"
        ELSE NULL
    END AS error_msg
FROM test_forecast_erp_data
WHERE allocated_qty IS NULL OR typeof(allocated_qty) <> "DECIMAL(38,2)"
;
SELECT
    txn_id,
    CASE
        WHEN delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8 THEN "delivery_dt must be yyyymmdd format"
        ELSE NULL
    END AS delivery_dt_error,
    CASE
        WHEN sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8 THEN "sched_dt must be yyyymmdd format"
        ELSE NULL
    END AS sched_dt_error
FROM test_forecast_erp_data
WHERE (delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8)
   OR (sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8)
;
SELECT
    txn_id,
    CASE
        WHEN flag_key IS NULL THEN "flag_key must be boolean"
        ELSE NULL
    END AS flag_key_error
FROM test_forecast_erp_data
WHERE flag_key IS NULL
;
SELECT
    txn_id,
    COUNT(*) AS cnt
FROM test_forecast_erp_data
WHERE txn_id IS NOT NULL
GROUP BY txn_id
HAVING cnt > 1
;
SELECT
    txn_id,
    SUM(COALESCE(allocated_qty, 0.00)) AS total_allocated_qty
FROM test_forecast_erp_data
WHERE txn_id = "TXN001"
GROUP BY txn_id
;
SELECT
    txn_id,
    CASE
        WHEN allocated_qty IS NULL THEN "allocated_qty must be decimal"
        WHEN typeof(allocated_qty) <> "DECIMAL(38,2)" THEN "allocated_qty must be decimal"
        WHEN delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8 THEN "delivery_dt must be yyyymmdd format"
        WHEN sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8 THEN "sched_dt must be yyyymmdd format"
        WHEN flag_key IS NULL THEN "flag_key must be boolean"
        ELSE NULL
    END AS error_message
FROM test_forecast_erp_data
WHERE
    allocated_qty IS NULL
    OR typeof(allocated_qty) <> "DECIMAL(38,2)"
    OR (delivery_dt IS NOT NULL AND LENGTH(CAST(delivery_dt AS STRING)) <> 8)
    OR (sched_dt IS NOT NULL AND LENGTH(CAST(sched_dt AS STRING)) <> 8)
    OR flag_key IS NULL
;
SELECT
    CASE
        WHEN typeof(txn_id) = "STRING"
         AND typeof(allocated_qty) = "DECIMAL(38,2)"
         AND typeof(delivery_dt) = "DECIMAL(38,0)"
         AND typeof(sched_dt) = "DECIMAL(38,0)"
         AND typeof(flag_key) = "BOOLEAN"
        THEN "PASS"
        ELSE "FAIL"
    END AS output_structure_validation
FROM test_forecast_erp_data
LIMIT 1
;
SELECT COUNT(*) AS total_test_rows FROM test_forecast_erp_data
;
SELECT
    txn_id,
    allocated_qty,
    flag_key,
    RANK() OVER (PARTITION BY flag_key ORDER BY allocated_qty DESC NULLS LAST) AS rank_allocated
FROM test_forecast_erp_data
ORDER BY flag_key DESC, rank_allocated ASC
;
CREATE TABLE IF NOT EXISTS purgo_playground.test_forecast_erp_delta (
    txn_id STRING,
    allocated_qty DECIMAL(38,2),
    delivery_dt DECIMAL(38,0),
    sched_dt DECIMAL(38,0),
    flag_key BOOLEAN,
    CONSTRAINT delivery_dt_format CHECK (LENGTH(CAST(delivery_dt AS STRING)) = 8),
    CONSTRAINT sched_dt_format CHECK (LENGTH(CAST(sched_dt AS STRING)) = 8),
    CONSTRAINT flag_key_bool CHECK (flag_key IN (TRUE, FALSE))
);

DELETE FROM purgo_playground.test_forecast_erp_delta
WHERE txn_id = "TXN002"
;

-- Validate Delta table row count after DML
SELECT COUNT(*) AS delta_table_row_count FROM purgo_playground.test_forecast_erp_delta
;

-- Cleanup: Drop test Delta table
DROP TABLE IF EXISTS purgo_playground.test_forecast_erp_delta
;
